# Neural Network Building for a Soccer Player Quality Estimation


## Import data and libraries

In [ ]:
!pip install -q gdown

import gdown
import os

def download_if_not_exists(file_id, output_name):
    if not os.path.exists(output_name):
        url = f"https://drive.google.com/uc?id={file_id}"
        gdown.download(url, output_name, quiet=False)
    else:
        print(f"{output_name} already downloaded")

# Data files download
download_if_not_exists("1tDFqS0OeTl861QqLW46BX0wquOC7Zvx0", "att.csv")
download_if_not_exists("1gDF3UGIDYOnFTpyF3UDLAET42H5obMm0", "labels.csv")

# Callbacks download
download_if_not_exists("1tM_IWIZB9qcFdb8Vffj2NVvdvT6tO5E6", "my_callbacks.py")

In [ ]:
# Tensorflow and tf.keras
import tensorflow as tf
from tensorflow import keras
print("Tensorflow version:",tf.__version__)
print("Keras version:",keras.__version__)

# Helper libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix

# Callbacks
from my_callbacks import TqdmProgressCallback, StopWhenValLossBelow, StopTrainingAtEpoch
from keras.callbacks import EarlyStopping

## Load data

In [ ]:
TRAIN_RATE = 0.8

attributes = pd.read_csv("att.csv")
label = pd.read_csv("labels.csv")

n_instances = attributes.shape[0]
n_train = int(n_instances*TRAIN_RATE)
n_dev = int((n_instances-n_train)/2)

x_train = attributes.values[:n_train]
t_train = label.values[:n_train]

x_dev = attributes.values[n_train:n_train+n_dev]
t_dev = label.values[n_train:n_train+n_dev]

print ("x_train:",x_train.shape)
print ("t_train:",t_train.shape)

print ("x_dev:",x_dev.shape)
print ("t_dev:",t_dev.shape)

## Initialize variables

In [ ]:
INPUTS = x_train.shape[1]
OUTPUTS = t_train.shape[1]
NUM_TRAINING_EXAMPLES = int(round(x_train.shape[0]/1))
NUM_DEV_EXAMPLES = int (round (x_dev.shape[0]/1))

In [ ]:
x_train[:5]

In [ ]:
t_train[:10]

In [ ]:
x_dev[:5]

In [ ]:
t_dev[:5]

## Set hyperparameters

 A 22-X-X-X-4 full-connected deep neural network architecture

In [ ]:
n_epochs = 450
lr = 0.0004
batch_size = 512
n_neurons_per_hlayer = [64, 32, 16]

## Build the deep neural model

Using **the functional API**, the Input layer defines the shape of each data element to expect as a tuple.
* The Input layer defines the shape of each data element to expect as a tuple.
* The Dense layers contain a number of neurons that are densely connected to the previous layer (every unit in the layer is connected to every unit in the previous layer).

In [ ]:
input_layer = keras.layers.Input(shape=(INPUTS,))
x = input_layer

# With L1
"""
for neurons in n_neurons_per_hlayer:
  x = keras.layers.Dense(units=neurons, activation = 'relu', kernel_initializer=keras.initializers.HeNormal(seed=123),
                         kernel_regularizer=keras.regularizers.l1(0.01))(x)
"""

# With L2
"""
for neurons in n_neurons_per_hlayer:
  x = keras.layers.Dense(units=neurons, activation = 'relu', kernel_initializer=keras.initializers.HeNormal(seed=123),
                         kernel_regularizer=keras.regularizers.l2(0.005))(x)
"""

# With L2 and Leaky ReLU
"""
for neurons in n_neurons_per_hlayer:
  x = keras.layers.Dense(units=neurons, activation = None, kernel_initializer=keras.initializers.HeNormal(seed=123),
                         kernel_regularizer=keras.regularizers.l2(0.005))(x)
  x = keras.layers.LeakyReLU(negative_slope=0.01)(x)
"""

# With L2 and ELU
"""
for neurons in n_neurons_per_hlayer:
  x = keras.layers.Dense(units=neurons, activation = 'elu', kernel_initializer=keras.initializers.HeNormal(seed=123),
                         kernel_regularizer=keras.regularizers.l2(0.005))(x)
"""

# With L1 and L2
"""
for neurons in n_neurons_per_hlayer:
  x = keras.layers.Dense(units=neurons, activation = 'relu', kernel_initializer=keras.initializers.HeNormal(seed=123),
                         kernel_regularizer=keras.regularizers.l1_l2(0.001))(x)
"""

# With dropout
dropout_rates = [0.1, 0.2, 0.1] # Define specific dropout rates for each hidden layer

for i, neurons in enumerate(n_neurons_per_hlayer):
  x = keras.layers.Dense(units=neurons, activation = 'relu', kernel_initializer=keras.initializers.HeNormal(seed=123))(x)
  x = keras.layers.Dropout(dropout_rates[i])(x)

# With dropout and batch normalization

"""
dropout_rates = [0.1, 0.2, 0.1]

for i, neurons in enumerate(n_neurons_per_hlayer):
    x = keras.layers.Dense(
        units=neurons,
        kernel_initializer=keras.initializers.HeNormal(seed=123)
    )(x)

    x = keras.layers.BatchNormalization()(x)
    x = keras.layers.Activation('relu')(x)

    x = keras.layers.Dropout(dropout_rates[i])(x)
"""

# With dropout and L2
"""
dropout_rates = [0.1, 0.2, 0.1] # Define specific dropout rates for each hidden layer

for i, neurons in enumerate(n_neurons_per_hlayer):
  x = keras.layers.Dense(units=neurons, activation = 'relu', kernel_initializer=keras.initializers.HeNormal(seed=123),
                         kernel_regularizer=keras.regularizers.l2(0.005))(x)
  x = keras.layers.Dropout(dropout_rates[i])(x)
"""

# With dropout and Leaky ReLU
"""
dropout_rates = [0.1, 0.2, 0.1] # Define specific dropout rates for each hidden layer

for i, neurons in enumerate(n_neurons_per_hlayer):
  x = keras.layers.Dense(units=neurons, activation = None, kernel_initializer=keras.initializers.HeNormal(seed=123))(x)
  x = keras.layers.Dropout(dropout_rates[i])(x)
  x = keras.layers.LeakyReLU(negative_slope=0.01)(x)
"""

# With dropout and ELU
"""
dropout_rates = [0.1, 0.2, 0.1] # Define specific dropout rates for each hidden layer

for i, neurons in enumerate(n_neurons_per_hlayer):
  x = keras.layers.Dense(units=neurons, activation = 'elu', kernel_initializer=keras.initializers.HeNormal(seed=123))(x)
  x = keras.layers.Dropout(dropout_rates[i])(x)
"""

# With batch normalization, L2 y dropout
"""
for neurons in n_neurons_per_hlayer:
    x = keras.layers.Dense(units=neurons, use_bias=False, kernel_initializer=keras.initializers.HeNormal(seed=123),
        kernel_regularizer=keras.regularizers.l2(0.005)
    )(x)
    x = keras.layers.BatchNormalization()(x)
    x = keras.layers.Activation('relu')(x)
    x = keras.layers.Dropout(0.3)(x)
"""

output_layer = keras.layers.Dense(units=OUTPUTS, activation = 'softmax')(x)
model = keras.models.Model(input_layer, output_layer, name='DeepFeedForward')
model.summary()


## Compile the model

Available loss functions, optimizers, and metrics at: https://keras.io/losses/, https://keras.io/optimizers/ and https://keras.io/metrics/.

In [ ]:
OPTIMIZER_SGD = tf.keras.optimizers.SGD(learning_rate = lr)
lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(initial_learning_rate=lr,decay_steps=50,decay_rate=0.98, staircase=True)
OPTIMIZER_Momentum = tf.keras.optimizers.SGD(
    learning_rate=tf.keras.optimizers.schedules.ExponentialDecay(initial_learning_rate=lr,decay_steps=100,decay_rate=0.98, staircase=True),
    momentum=0.9, nesterov=False)
OPTIMIZER_RMSprop = tf.keras.optimizers.RMSprop(learning_rate = lr, rho=0.9)
OPTIMIZER_Adam = tf.keras.optimizers.Adam(learning_rate = lr, beta_1=0.9, beta_2=0.999)
OPTIMIZER_Nadam = tf.keras.optimizers.Nadam(learning_rate = lr)

# Model SGD
"""
model.compile(loss=tf.keras.losses.categorical_crossentropy,
              optimizer=OPTIMIZER_SGD,
              metrics=["categorical_accuracy"])
"""

# Model Decay
"""
model.compile(loss=tf.keras.losses.categorical_crossentropy,
              optimizer=tf.keras.optimizers.SGD(learning_rate=lr_schedule),
              metrics=["categorical_accuracy"])
"""

# Model Momentum
""""
model.compile(loss=tf.keras.losses.categorical_crossentropy,
    optimizer=OPTIMIZER_Momentum,
    metrics=["categorical_accuracy"]
)
"""

# Model RMSProp
""""
model.compile(loss=tf.keras.losses.categorical_crossentropy,
    optimizer=OPTIMIZER_RMSprop,
    metrics=["categorical_accuracy"]
)
"""

# Model Adam
"""
model.compile(loss=tf.keras.losses.categorical_crossentropy,
    optimizer=OPTIMIZER_Adam,
    metrics=["categorical_accuracy"]
)
"""

# Model Nadam
model.compile(loss=tf.keras.losses.categorical_crossentropy,
    optimizer=OPTIMIZER_Nadam,
    metrics=["categorical_accuracy"]
)

## Train the model with M-BGD

In [ ]:
from keras.callbacks import EarlyStopping
patienceStop_callback = EarlyStopping(monitor='val_loss', patience=20)
# If the val_loss does not improve for 20 consecutive epochs, the training will be stopped early.

stopWhenValLossBelow_callback = StopWhenValLossBelow(threshold=0.65)
stopTrainingAtEpoch_callback = StopTrainingAtEpoch(stop_epoch=5)

tqdm_callback = TqdmProgressCallback(epochs=n_epochs)

import time
start = time.perf_counter()
history = model.fit(x_train, t_train, batch_size=batch_size, epochs=n_epochs, verbose=0, validation_data=(x_dev, t_dev),
                   callbacks=[tqdm_callback])
print (time.perf_counter() - start)

## Get the results

In [ ]:
results=pd.DataFrame(history.history)
results.plot(figsize=(8, 5))
plt.grid(True)
plt.xlabel ("Epochs")
plt.ylabel ("Accuracy - Mean Log Loss")
plt.gca().set_ylim(0, 1) # Set the vertical range to [0-1]
plt.show()

In [ ]:
history.params

In [ ]:
results[-1:]

In [ ]:
print ("Accuracy for the training set: ", results.categorical_accuracy.values[-1:][0])

In [ ]:
print ("Accuracy for the development test set: ", results.val_categorical_accuracy.values[-1:][0])

Let's see how the model predicts using the development test set:

In [ ]:
dev_predictions=model.predict(x_dev).round(2)
dev_predictions[:20]

In [ ]:
dev_rounded_predictions=np.round(dev_predictions)
indices = np.argmax(dev_predictions,1)
for row, index in zip(dev_rounded_predictions, indices): row[index]=1
dev_rounded_predictions[:20]

In [ ]:
t_dev[:20] #target classes

In [ ]:
dev_correct_predictions = np.equal(np.argmax(dev_rounded_predictions,1),np.argmax(t_dev,1))
print (dev_correct_predictions[:30])

In [ ]:
from collections import Counter
Counter (dev_correct_predictions)

Loss promedio de cada clase predicha, tanto en TRAIN como en DEV

In [ ]:
y_pred = model.predict(x_train)
y_true = t_train

pred_clase = tf.argmax(y_pred, axis=1)

loss_fn = tf.keras.losses.CategoricalCrossentropy(reduction="none")
losses = loss_fn(y_true, y_pred)

pred_clase = pred_clase.numpy()
losses = losses.numpy()

n_clases = np.max(pred_clase) + 1

loss_medio_por_clase = [
    losses[pred_clase == c].mean() if np.any(pred_clase == c) else np.nan
    for c in range(n_clases)
]

print(loss_medio_por_clase)

In [ ]:
y_pred = model.predict(x_dev)
y_true = t_dev

pred_clase = tf.argmax(y_pred, axis=1)

loss_fn = tf.keras.losses.CategoricalCrossentropy(reduction="none")
losses = loss_fn(y_true, y_pred)

pred_clase = pred_clase.numpy()
losses = losses.numpy()

n_clases = np.max(pred_clase) + 1

loss_medio_por_clase = [
    losses[pred_clase == c].mean() if np.any(pred_clase == c) else np.nan
    for c in range(n_clases)
]

print(loss_medio_por_clase)

## Final Test

We are suppossing that this is the final model that achieves the best performance.

### Get the final test set

In [ ]:
n_final_test = n_instances-n_train-n_dev

x_final_test = attributes.values[n_train+n_dev:n_instances]
t_final_test = label.values[n_train+n_dev:n_instances]

print ("x_test:",x_final_test.shape)
print ("t_test:",t_final_test.shape)

### Evaluate the model

In [ ]:
model.evaluate(x_final_test, t_final_test)

Predictions on the final test set:

In [ ]:
test_predictions=model.predict(x_final_test)
test_rounded_predictions=np.round(test_predictions)
indices = np.argmax(test_predictions,1)
for row, index in zip(test_rounded_predictions, indices): row[index]=1
test_rounded_predictions[:20]

The target outputs:

In [ ]:
t_final_test[:20]

The first 30 predictions. True means that the neural network correctly classifies the input vector.  

In [ ]:
test_correct_predictions = np.equal(np.argmax(test_rounded_predictions,1),np.argmax(t_final_test,1))
test_correct_predictions[:30]

The final test accuracy:

In [ ]:
from collections import Counter
final_test_prediction_results=Counter(test_correct_predictions)
final_test_prediction_results

In [ ]:
final_test_prediction_results[True]/sum(final_test_prediction_results.values())

### Confusion Matrix on the final test set

In [ ]:
test_sparse_predictions = np.argmax (test_predictions,axis=1)
test_sparse_targets = np.argmax(t_final_test, axis=1)
print('Confusion Matrix')
c_m=pd.DataFrame(confusion_matrix(test_sparse_predictions,test_sparse_targets))
c_m['Sum']=c_m.sum(axis=1, numeric_only=True)
c_m

### Clasification metrics

In [ ]:
print('Classification Report')
classes = ["1", "2", "3", "4"]
print(classification_report(test_sparse_targets, test_sparse_predictions, target_names=classes))

In [ ]:
y_score = model.predict(x_final_test)

y_true = t_final_test

n_classes = y_true.shape[1]

import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc

plt.figure(figsize=(8,6))
for i in range(n_classes):
    fpr, tpr, _ = roc_curve(y_true[:, i], y_score[:, i])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f'Clase {i} (AUC = {roc_auc:.2f})')

plt.plot([0, 1], [0, 1], 'k--', label='Aleatorio')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Curvas ROC - Conjunto de Test')
plt.legend(loc='lower right')
plt.show()

In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

y_pred = np.argmax(model.predict(x_final_test), axis=1)

y_true = np.argmax(t_final_test, axis=1)

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=True)
plt.xlabel('Predicción')
plt.ylabel('Verdadero')
plt.title('Matriz de Confusión - Conjunto de Test')
plt.show()

In [ ]:
y_pred = model.predict(x_final_test)
y_true = t_final_test

pred_clase = tf.argmax(y_pred, axis=1)

loss_fn = tf.keras.losses.CategoricalCrossentropy(reduction="none")
losses = loss_fn(y_true, y_pred)

pred_clase = pred_clase.numpy()
losses = losses.numpy()

n_clases = np.max(pred_clase) + 1

loss_medio_por_clase = [
    losses[pred_clase == c].mean() if np.any(pred_clase == c) else np.nan
    for c in range(n_clases)
]

print(loss_medio_por_clase)

In [ ]:
y_pred = np.argmax(model.predict(x_train), axis=1)

y_true = np.argmax(t_train, axis=1)

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', cbar=True)
plt.xlabel('Predicción')
plt.ylabel('Verdadero')
plt.title('Matriz de Confusión - Conjunto de Train')
plt.show()

In [ ]:
test_sparse_predictions = np.argmax (model.predict(x_train),axis=1)
test_sparse_targets = np.argmax(t_train, axis=1)

print('Classification Report')
classes = ["1", "2", "3", "4"]
print(classification_report(test_sparse_targets, test_sparse_predictions, target_names=classes))